# Chain of Thought (CoT) with LangChain

This notebook demonstrates how to implement Chain of Thought prompting using LangChain libraries. Chain of Thought is a technique that encourages language models to reason step by step, leading to more accurate and explainable outputs.

## What is Chain of Thought?

Chain of Thought (CoT) is a prompting strategy where you instruct the language model to break down its reasoning into intermediate steps. Instead of jumping directly to an answer, the model is encouraged to "think step by step" like a human would.

### Benefits of CoT:
- **Improved accuracy**: Step-by-step reasoning reduces errors
- **Better explainability**: You can see the model's thought process
- **Handles complex tasks**: Useful for math, logic, and multi-step problems

### How it works:
1. Include phrases like "Let's think step by step" or "Reason step by step" in your prompt
2. Provide examples of step-by-step reasoning (few-shot prompting)
3. The model generates intermediate reasoning before the final answer

In [4]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

# Load environment variables
load_dotenv()

# Set up OpenAI API key (make sure to set OPENAI_API_KEY in your .env file)
# For this example, we'll use a simple setup. In practice, you'd want to handle API keys securely.

# Note: If you don't have an OpenAI API key, you can modify this to use a different LLM
# For example, you could use Ollama with local models or other providers

c:\Users\dell\Documents\Agentic AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
# Initialize the language model
# Note: You need to set your OPENAI_API_KEY environment variable
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.7  # Some creativity for reasoning
)

In [ ]:
# Create a Chain of Thought prompt template zero shot
cot_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are a helpful assistant that solves problems using Chain of Thought reasoning.

Question: {question}

Please solve this problem by thinking step by step. Show your reasoning process clearly, then provide the final answer.

Step-by-step reasoning:
1. First, understand what the question is asking
2. Break down the problem into smaller parts
3. Apply logical reasoning to each part
4. Combine the results
5. Arrive at the final answer

Final Answer:"""
)

In [7]:
# Create the Chain of Thought chain using RunnableSequence (modern LangChain approach)
cot_chain = cot_prompt | llm

In [8]:
# Example 1: A simple math word problem
question1 = "If a train travels at 60 miles per hour for 2.5 hours, how far does it travel?"

print("Question:", question1)
print("\nChain of Thought Response:")
response1 = cot_chain.invoke({"question": question1})
print(response1.content)

Question: If a train travels at 60 miles per hour for 2.5 hours, how far does it travel?

Chain of Thought Response:
The train travels at 60 miles per hour for 2.5 hours. To find out how far it travels, we need to multiply the speed of the train (60 miles per hour) by the time it travels (2.5 hours).

60 miles per hour * 2.5 hours = 150 miles

Therefore, the train travels 150 miles.


In [9]:
# Example 2: A logic puzzle
question2 = "A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?"

print("\n" + "="*50)
print("Question:", question2)
print("\nChain of Thought Response:")
response2 = cot_chain.invoke({"question": question2})
print(response2.content)


Question: A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?

Chain of Thought Response:
The ball costs $0.05. 

Reasoning:
1. The total cost of the bat and the ball is $1.10.
2. Let x be the cost of the ball.
3. The bat costs $1.00 more than the ball, so the cost of the bat is x + $1.00.
4. The total cost is x (ball) + (x + $1.00) (bat) = $1.10.
5. This equation can be written as 2x + $1.00 = $1.10.
6. Solving for x, we get 2x = $0.10, x = $0.05.
7. Therefore, the ball costs $0.05.


In [10]:
# Example 3: Tree of Thought (exploring multiple reasoning paths)
# Tree of Thought extends CoT by exploring multiple reasoning branches simultaneously
# and selecting the best reasoning path

tree_of_thought_prompt = PromptTemplate(
    input_variables=["question"],
    template="""You are a helpful assistant that solves problems using Tree of Thought reasoning.

Question: {question}

Please solve this problem by exploring multiple reasoning paths:

1. First, generate 3 different approaches to solve this problem:
   Path A: [First approach]
   Path B: [Second approach]
   Path C: [Third approach]

2. For each path, think through the reasoning step by step

3. Evaluate which path is most logical and reliable

4. Follow the best path to arrive at the final answer

Reasoning:
Path A reasoning:
[detailed steps]

Path B reasoning:
[detailed steps]

Path C reasoning:
[detailed steps]

Evaluation: [Which path is best and why?]

Final Answer: [Based on the best reasoning path]"""
)

tot_chain = tree_of_thought_prompt | llm

# Example 3: A complex problem that benefits from exploring multiple paths
question3 = "A company has three production lines. Line A produces 100 units/day, Line B produces 150 units/day, and Line C is down for maintenance. If they need to produce 500 units in 3 days, can they meet the deadline? Consider factors like quality control and buffer time."

print("\n" + "="*50)
print("Question:", question3)
print("\nTree of Thought Response:")
response3 = tot_chain.invoke({"question": question3})
print(response3.content)


Question: A company has three production lines. Line A produces 100 units/day, Line B produces 150 units/day, and Line C is down for maintenance. If they need to produce 500 units in 3 days, can they meet the deadline? Consider factors like quality control and buffer time.

Tree of Thought Response:
Path A reasoning:
1. Calculate the total production capacity per day: 100 units/day (Line A) + 150 units/day (Line B) = 250 units/day
2. Calculate the total production capacity over 3 days: 250 units/day * 3 days = 750 units
3. Since the company needs to produce 500 units in 3 days, they can meet the deadline with a buffer of 250 units.

Path B reasoning:
1. Consider the downtime of Line C for maintenance - if Line C is down for the entire 3 days, the company will only have a production capacity of 250 units/day (Line A + Line B)
2. Calculate the total production capacity over 3 days: 250 units/day * 3 days = 750 units
3. Since the company needs to produce 500 units in 3 days, they can mee

## Tree of Thought (ToT) Reasoning

Tree of Thought is an advanced reasoning technique that extends Chain of Thought by exploring multiple reasoning paths simultaneously instead of following a single linear path.

### Key Differences from CoT:
- **CoT**: Follows one reasoning path from start to finish
- **ToT**: Explores multiple branches and selects the best path

### How Tree of Thought Works:
1. Generate multiple candidate approaches or reasoning paths for a problem
2. Evaluate each path independently with step-by-step reasoning
3. Score or compare the quality of each reasoning path
4. Select and follow the best reasoning path to the solution

### When to Use Tree of Thought:
- Complex problems with multiple solution approaches
- Situations where initial path might lead to suboptimal solutions
- Problems requiring exploration and evaluation of alternatives
- Decision-making scenarios with trade-offs

### Benefits of ToT over CoT:
- Reduces likelihood of getting stuck in flawed reasoning
- Explores solution space more thoroughly
- Better for problems with multiple valid approaches
- Can identify the most robust solution path

## Understanding the Examples

### Example 1: Math Word Problem
This demonstrates how CoT helps break down arithmetic problems into understandable steps.

### Example 2: Logic Puzzle
This classic puzzle shows how CoT prevents common mistakes. Many people initially think the ball costs $0.10, but that would make the bat cost $1.10, totaling $1.20 - which is wrong!

The correct answer requires recognizing that the bat costs $1.05 and the ball costs $0.05.

### Example 3: Tree of Thought - Complex Decision Problem
This demonstrates how exploring multiple reasoning paths can lead to better solutions for complex problems. Instead of just following one reasoning chain, the model evaluates different approaches and selects the most robust one. This is particularly useful for real-world business problems with multiple constraints and trade-offs.

## Key Takeaways

1. **Prompt Design**: Include explicit instructions to "think step by step"
2. **Structured Reasoning**: Guide the model through logical steps
3. **Multiple Paths**: Use Tree of Thought for complex problems requiring exploration
4. **Few-shot Learning**: You can include examples in the prompt for better results
5. **Error Prevention**: CoT/ToT helps avoid impulsive or incorrect answers

## Try It Yourself!

Modify the `question` variable above with your own problems and see how the model reasons through them. Try comparing how CoT vs ToT handles your problem!

## Next Steps and Advanced Topics

### 1. Few-Shot Prompting
Enhance CoT by including examples in your prompt:

```
template = """
Solve problems using Chain of Thought.

Example:
Question: What is 15 + 27?
Reasoning: 15 + 20 = 35, 35 + 7 = 42
Answer: 42

Now solve:
Question: {question}
Reasoning:
"""
```

### 2. Self-Consistency
Run the same prompt multiple times and take the majority answer for better accuracy.



### 4. Experiment with Different Models
Try different LLMs to see how they handle CoT. Some models are better at step-by-step reasoning than others.

### 5. Custom CoT Prompts
Adapt the prompt template for specific domains (math, coding, logic puzzles, etc.)

Happy learning! 🚀

In [11]:
# Example 4: Tree of Thought on a web document
# Load a web page using Tavily search, extract the text, and analyze it using multiple reasoning paths.
import requests
import re
from html import unescape
from tavily import TavilyClient
import os

def html_to_text(html: str) -> str:
    clean = re.sub(r'(?is)<(script|style).*?>.*?</(script|style)>', ' ', html)
    clean = re.sub(r'<[^>]+>', ' ', clean)
    clean = unescape(clean)
    clean = re.sub(r'\s+', ' ', clean)
    return clean.strip()

# Search for a relevant document using Tavily
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
search_query = "artificial intelligence recent developments 2024"
print("Searching for:", search_query)
search_results = tavily_client.search(search_query, max_results=1)
if search_results['results']:
    url = search_results['results'][0]['url']
    print("Found document URL:", url)
    response = requests.get(url, timeout=10)
    web_text = html_to_text(response.text)
    web_excerpt = " ".join(web_text.split()[:400])
else:
    print("No results found, using fallback.")
    url = "https://www.example.com"
    response = requests.get(url, timeout=10)
    web_text = html_to_text(response.text)
    web_excerpt = " ".join(web_text.split()[:400])

web_tree_prompt = PromptTemplate(
    input_variables=["document_excerpt"],
    template="""You are a helpful assistant that solves problems using Tree of Thought reasoning.

Document Excerpt:
{document_excerpt}

Please analyze this excerpt by exploring multiple reasoning paths:

1. Generate 3 distinct analytical approaches:
   Path A: [Overview and main message]
   Path B: [Critical strengths and weaknesses]
   Path C: [Practical implications and next steps]

2. For each path, think through the reasoning step by step.

3. Compare the paths and decide which one gives the best insight.

4. Produce a concise final analysis based on the strongest path.

Reasoning:
Path A reasoning:
[detailed steps]

Path B reasoning:
[detailed steps]

Path C reasoning:
[detailed steps]

Evaluation: [Which path is best and why?]

Final Analysis:"""
)

web_tree_chain = web_tree_prompt | llm
print("\nExcerpt preview:\n", web_excerpt[:350], "...\n")
print("Tree of Thought Response:")
response4 = web_tree_chain.invoke({"document_excerpt": web_excerpt})
print(response4.content)

Searching for: artificial intelligence recent developments 2024
Found document URL: https://www.linkedin.com/pulse/state-ai-2024-key-trends-innovations-watch-aierteam-6rmoc

Excerpt preview:
 The State of AI in 2024: Key Trends and Innovations to Watch Agree & Join LinkedIn By clicking Continue to join or sign in, you agree to LinkedIn’s User Agreement , Privacy Policy , and Cookie Policy . Skip to main content LinkedIn Top Content People Learning Jobs Games Sign in Join now The State of AI in 2024: Key Trends and Innovations to Watch R ...

Tree of Thought Response:
Path A reasoning:
1. Overview: The document discusses the current state of AI in 2024, highlighting key trends and innovations shaping the AI landscape.
2. Main Message: AI, particularly generative AI, AI ethics, and AI in healthcare, are key areas driving technological advancements and societal impact.

Path B reasoning:
1. Critical Strengths: The document effectively identifies the significant trends in AI, such as the g

### 3. Integration with Other LangChain Components
- Use CoT with document loaders for reasoning over documents
- Combine with memory for conversational CoT
- Integrate with tools for multi-step agent reasoning